# DVD-JEPA — train a real JEPA world model in ~10 seconds

A **Joint-Embedding Predictive Architecture** world model for a bouncing DVD logo.
It predicts the future *in representation space*, learns the physics with no labels,
dreams the future once you add a decoder, and doubles as an anomaly detector.

▶ **Live browser demo:** https://dvd-jepa.vercel.app  ·  **Repo:** https://github.com/mandarwagh9/dvd-jepa

Runs on CPU — no GPU runtime needed.

In [ ]:
# Get the code (torch is preinstalled on Colab)
!git clone -q https://github.com/mandarwagh9/dvd-jepa.git
%cd dvd-jepa
import torch, numpy as np, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
print('torch', torch.__version__)

## 1. The world

A DVD logo bouncing in a 16×16 box. The future is unreadable from one frame
(you can't see velocity) but perfectly predictable from two — so an *observation*
is a stack of two consecutive frames.

In [ ]:
from dvd_jepa.world import make_sequences, build_pairs, roll_one, H, W
frames, pos = make_sequences(n_seq=4, T=64, seed=1)
fig, ax = plt.subplots(1, 8, figsize=(12, 1.8))
for i in range(8):
    ax[i].imshow(frames[0, i*4], cmap='magma'); ax[i].axis('off'); ax[i].set_title(f't={i*4}', fontsize=8)
plt.suptitle('one bouncing-logo sequence', y=1.15); plt.show()

## 2. Train the JEPA

Online encoder + EMA target encoder (stop-grad) + a predictor that matches the
future embedding, plus a VICReg variance term so the representation can't collapse.
Watch `emb-std`: it starts near 0 (collapsing) and is pushed up to ~3 (healthy).

In [ ]:
from dvd_jepa.train import train_jepa, train_decoder, linear_probe, forecast, anomaly_scan
frames, _ = make_sequences(seed=0)
obs, nxt = build_pairs(frames)
online, target, predictor = train_jepa(obs, nxt, steps=2500)
dec = train_decoder(target, obs, steps=2000)

## 3. Did it actually learn the world?

Freeze the encoder and fit a **linear** map from the 32-d latent to the true (y, x)
position. Low error ⇒ the latent encodes exact world state, despite never being
told a coordinate.

In [ ]:
# build position labels (latest frame of each observation)
_, pos = make_sequences(seed=0)
lbl = torch.tensor(np.concatenate([pos[:, t+1] for t in range(frames.shape[1]-2)]), dtype=torch.float32)
_, rmse = linear_probe(target, obs, lbl)
print(f'linear probe recovers position to {rmse:.2f} px in a {H}x{W} box')

## 4. Make it dream (forecast + decode)

Seed from 2 frames, roll the predictor forward **in latent space**, decode each
step to pixels. White = reality, cyan = the dream. It tracks for ~20 steps,
including the wall bounce, then drifts (single-step predictor, errors compound).

In [ ]:
vf, _ = roll_one(T=44, seed=7)
preds, errs = forecast(target, predictor, dec, vf, K=24)
fig, ax = plt.subplots(2, 12, figsize=(13, 2.4))
for k in range(12):
    ax[0, k].imshow(vf[2*k+2], cmap='gray'); ax[0, k].axis('off')
    ax[1, k].imshow(preds[2*k], cmap='gray'); ax[1, k].axis('off')
ax[0, 0].set_ylabel('reality', rotation=0, labelpad=28); ax[1, 0].set_ylabel('dream', rotation=0, labelpad=24)
plt.suptitle(f'future-frame prediction (mean MSE {np.mean(errs):.4f})'); plt.show()

## 5. Use it: predictive anomaly detection

Run it as a 1-step monitor. When reality stops matching the rendered expectation,
prediction error spikes — a usable anomaly signal. Inject a teleport and watch.

In [ ]:
af, _ = roll_one(T=44, teleport_at=24, seed=3)
surprise = anomaly_scan(target, predictor, dec, af)
plt.figure(figsize=(8, 3))
plt.plot(surprise, color='#2bd4ff', lw=2, label='surprise')
plt.axvline(24, color='#ffce54', ls=':', lw=2, label='injected teleport')
plt.title('predictive surprise'); plt.xlabel('frame'); plt.legend(); plt.show()
print(f'peak surprise is {surprise.max()/np.median(surprise):.0f}x the baseline')

## 6. The pure JEPA (the joke)

A *pure* JEPA has **no decoder** — it only outputs vectors. Here is the model's
entire mental state for one observation. It understands the bounce perfectly and
cannot draw a single pixel of it:

In [ ]:
z = target(obs[:1]).detach().numpy()[0]
print('the JEPA\'s 32-d world-state vector:\n', np.round(z, 2))
print('\n...which means, in the deepest abstract sense: "yeah. still bouncing."')

---
**Now go play with the live, interactive version:** https://dvd-jepa.vercel.app
— toggle the decoder off to see the pure-JEPA vectors, inject anomalies, and watch
it dream 30 steps ahead.  ·  ⭐ the repo: https://github.com/mandarwagh9/dvd-jepa